In [14]:
import pickle
import pandas as pd
import os


In [11]:
with open("model/model.bin", "rb") as f_in:
    dv, model = pickle.load(f_in)

c:\Users\AvadhootHublikar\Documents\mlops\exp-tracking\Lib\site-packages\sklearn\base.py:463: InconsistentVersionWarning: Trying to unpickle estimator DictVectorizer from version 1.5.0 when using version 1.8.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
c:\Users\AvadhootHublikar\Documents\mlops\exp-tracking\Lib\site-packages\sklearn\base.py:463: InconsistentVersionWarning: Trying to unpickle estimator LinearRegression from version 1.5.0 when using version 1.8.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


In [12]:
categorical = ['PULocationID', 'DOLocationID']

def read_data(filename):
    df = pd.read_parquet(filename)

    df['duration'] = df.tpep_dropoff_datetime - df.tpep_pickup_datetime
    df['duration'] = df.duration.dt.total_seconds() / 60

    df = df[(df.duration >= 1) & (df.duration <= 60)]
    df[categorical] = df[categorical].fillna(-1).astype('int').astype('str')

    return df

In [13]:
df = read_data('./data/yellow_tripdata_2023-03.parquet')
dicts = df[categorical].to_dict(orient='records')
X_val = dv.transform(dicts)
y_pred = model.predict(X_val)
print(y_pred.std())

6.247488852238704


In [17]:
year = 2023
month = 3

df['ride_id'] = f'{year:04d}/{month:02d}_' + df.index.astype('str')

df_result = pd.DataFrame({
    'ride_id': df['ride_id'],
    'predicted_duration': y_pred
})

output_file = './data/yellow_tripdata_2023-03_predictions.parquet'

df_result.to_parquet(
    output_file,
    engine='pyarrow',
    index=False,
    compression=None
)

os.path.getsize(output_file) / 1024 / 1024

65.38200759887695